# 05 — SA Calibration (Phase 2 reproduction)

Reproduces the two-stage calibration of the SA method's free parameters,
using **only the 12 calibration instances** (never the 51 held-out
instances):

- **5A**: T0/cooling audit -- scale-aware grid search, confirming a
  flat/robust performance region (not a sharp single optimum).
- **5B**: lambda_s sweep over `{0,5,10,20,40,80,120}`, with a
  PROGRAMMATIC selection rule (harm=0% -> unnecessary=0% -> retention
  >=90% -> maximize RSI among feasible candidates).

**PUBLIC mode** reproduces the calibration SUMMARY and SELECTION
directly from the published, de-identified calibration outputs
(`data_deidentified/experiment_outputs/sa_t0_cooling_grid.csv` and
`sa_lambda_sweep.csv`) -- this is genuine statistical reproduction of the
real manuscript calibration data, not a computation against synthetic
data. Zero rows loaded from either published file is a HARD FAILURE, not
a silently-tolerated edge case.

**PRIVATE mode** performs the full computational calibration (the actual
grid search, via `src.search`) against authorized real operational
inputs.


In [ ]:
import os, sys, json, time
import pandas as pd
import numpy as np
assert 'REPO_ROOT' in dir(), "Run notebook 00 first."
sys.path.insert(0, REPO_ROOT)

with open(os.path.join(REPO_ROOT, "configs", "sa_config.json")) as f:
    sa_config = json.load(f)


## PUBLIC mode: statistical reproduction from published calibration outputs

In [ ]:
if DATA_MODE == "public":
    EXP_DIR = os.path.join(REPO_ROOT, "data_deidentified", "experiment_outputs")
    t0_path = os.path.join(EXP_DIR, "sa_t0_cooling_grid.csv")
    lambda_path = os.path.join(EXP_DIR, "sa_lambda_sweep.csv")

    t0_df = pd.read_csv(t0_path)
    lambda_df = pd.read_csv(lambda_path)

    print(f"Loaded {t0_path}: {len(t0_df)} rows")
    print(f"Loaded {lambda_path}: {len(lambda_df)} rows")

    # HARD FAIL on zero rows -- a silently-empty calibration table must
    # never be treated as an acceptable edge case.
    if len(t0_df) == 0:
        raise ValueError(f"HARD FAIL: {t0_path} loaded with zero calibration blocks/rows.")
    if len(lambda_df) == 0:
        raise ValueError(f"HARD FAIL: {lambda_path} loaded with zero calibration blocks/rows.")

    print("\n--- T0/cooling grid (published) ---")
    print(t0_df.to_string(index=False))
    print("\n--- lambda_s sweep (published) ---")
    print(lambda_df.to_string(index=False))


## PUBLIC mode: programmatic selection reproduction on the published data

In [ ]:
if DATA_MODE == "public":
    spread_pct = 100 * (t0_df['mean_lateness'].max() - t0_df['mean_lateness'].min()) / t0_df['mean_lateness'].mean()
    print(f"T0/cooling performance spread across grid: {spread_pct:.2f}% "
          f"(a small spread supports a 'flat region' finding, not a sharp optimum)")
    print(f"Frozen selection (configs/sa_config.json): T0={sa_config['T0']}, cooling={sa_config['cooling']}")

    step1 = lambda_df[lambda_df['harm_count'] == 0]
    step2 = step1[step1['unnecessary_count'] == 0]
    step3 = step2[step2['retention_pct'] >= 90]
    if len(step3) == 0:
        raise ValueError("HARD FAIL: no lambda_s candidate in the published sweep clears the "
                          "harm=0/unnecessary=0/retention>=90% selection rule -- this should never "
                          "happen against the real manuscript calibration data.")
    selected_row = step3.loc[step3['mean_rsi'].idxmax()]
    selected_lambda_s = int(selected_row['lambda_s'])
    print(f"\nSelected lambda_s = {selected_lambda_s} (retention={selected_row['retention_pct']:.2f}%, "
          f"RSI={selected_row['mean_rsi']:.4f}) -- reproduced programmatically from the published sweep.")
    print(f"Frozen (configs/sa_config.json): lambda_s={sa_config['lambda_s']}")
    if selected_lambda_s != sa_config['lambda_s']:
        raise AssertionError(f"Reproduced selection ({selected_lambda_s}) does not match the frozen "
                              f"config value ({sa_config['lambda_s']}) -- investigate root cause.")


## PRIVATE mode: full computational calibration

Runs the actual T0/cooling grid search and lambda_s sweep (via
`src.search.corrective_search_one_vehicle`) against authorized real
operational inputs. Not exercised in PUBLIC mode -- see above.


In [ ]:
if DATA_MODE == "private":
    from src.data import build_canonical_mapping, load_distance_matrix, load_travel_time_p50_matrix
    from src.simulator import SimulationContext, simulate_mixed_route, route_stability, objective_Z
    from src.search import corrective_search_one_vehicle
    from src.quantiles import P85_MULTIPLIER, P95_MULTIPLIER

    PRIVATE_DIR = os.path.join(REPO_ROOT, "data_private")
    required = ["customer_day_stops_preprocessed.csv", "combined_osrm_distance_matrix_km_long.csv",
                "combined_travel_time_matrix_p50_long.csv", "calibration_split.json", "full_sequences.csv"]
    missing = [f for f in required if not os.path.exists(os.path.join(PRIVATE_DIR, f))]
    if missing:
        raise FileNotFoundError(f"DATA_MODE=private requires authorized operational inputs; "
                                 f"missing: {missing}. See README Level 2.")

    mapping_df, customers, _ = build_canonical_mapping(os.path.join(PRIVATE_DIR, "customer_day_stops_preprocessed.csv"))
    parent_of = dict(zip(zip(mapping_df['delivery_date'], mapping_df['virtual_stop_id']), mapping_df['parent_physical_node']))
    dist_matrix = load_distance_matrix(os.path.join(PRIVATE_DIR, "combined_osrm_distance_matrix_km_long.csv"))
    p50_matrix = load_travel_time_p50_matrix(os.path.join(PRIVATE_DIR, "combined_travel_time_matrix_p50_long.csv"))
    ctx = SimulationContext(dist_matrix, p50_matrix, customers, parent_of, P85_MULTIPLIER, P95_MULTIPLIER)

    with open(os.path.join(PRIVATE_DIR, "calibration_split.json")) as f:
        split = json.load(f)
    calib_dates = sorted(split['calibration_dates'])
    seq_all = pd.read_csv(os.path.join(PRIVATE_DIR, "full_sequences.csv"))

    TRIGGERS = [0.25, 0.50, 0.75]
    SHOCKS = ['p85', 'p95', 'delay20', 'delay30']
    CALIB_SEED = 101

    calib_dates_run = calib_dates[:2] if RUN_MODE == "quick" else calib_dates
    TRIGGERS_run = TRIGGERS[:1] if RUN_MODE == "quick" else TRIGGERS
    SHOCKS_run = SHOCKS[:2] if RUN_MODE == "quick" else SHOCKS
    print(f"[{RUN_MODE.upper()} MODE] {len(calib_dates_run)} instances x {len(TRIGGERS_run)} trigger(s) x "
          f"{len(SHOCKS_run)} shock(s)")

    def build_vehicle_blocks(dates, triggers, shocks):
        blocks = []
        for date in dates:
            for trig in triggers:
                for shock in shocks:
                    block = seq_all[(seq_all['delivery_date']==date)&(seq_all['method']=='NR')&
                                     (seq_all['seed']==CALIB_SEED)&(seq_all['shock']==shock)&
                                     (seq_all['trigger_fraction']==trig)]
                    if block.empty:
                        continue
                    for vid, vgrp in block.groupby('vehicle_id'):
                        vgrp = vgrp.sort_values('sequence')
                        full_seq = vgrp['node_id'].tolist()
                        frozen_flags = vgrp['served_frozen'].tolist()
                        blocks.append((date, trig, shock, vid, full_seq, frozen_flags))
        return blocks

    calib_blocks = build_vehicle_blocks(calib_dates_run, TRIGGERS_run, SHOCKS_run)
    print(f"Total (instance,trigger,shock,vehicle) evaluation blocks: {len(calib_blocks)}")
    if len(calib_blocks) == 0:
        raise ValueError("HARD FAIL: zero calibration blocks constructed from private inputs.")

    def run_grid_point(T0, cooling, blocks):
        trial_config = dict(sa_config); trial_config['T0'] = T0; trial_config['cooling'] = cooling
        lateness_total, n_vehicles, harm_count = 0.0, 0, 0
        for date, trig, shock, vid, full_seq, frozen_flags in blocks:
            unserved_ct = sum(1 for f in frozen_flags if not f)
            m_nr = simulate_mixed_route(ctx, date, full_seq, frozen_flags, shock)
            if unserved_ct < 2:
                lateness_total += m_nr['lateness']; continue
            n_vehicles += 1
            res = corrective_search_one_vehicle(ctx, date, full_seq, frozen_flags, shock, 'SA',
                                                  trial_config, seed=CALIB_SEED)
            lateness_total += res['metrics']['lateness']
            if res['metrics']['lateness'] > m_nr['lateness'] + 1e-6:
                harm_count += 1
        return dict(T0=T0, cooling=cooling, mean_lateness=lateness_total/max(n_vehicles,1),
                    n_vehicles=n_vehicles, harm_count=harm_count)

    T0_GRID_PRIVATE = [sa_config['T0']] if RUN_MODE == "quick" else [25, 50, 75, sa_config['T0'], 150, 259, 521, 1209]
    COOLING_GRID_PRIVATE = [sa_config['cooling']] if RUN_MODE == "quick" else [0.90, 0.95, 0.98]
    t0_df_private = pd.DataFrame([run_grid_point(t0, c, calib_blocks) for t0 in T0_GRID_PRIVATE for c in COOLING_GRID_PRIVATE])
    print(t0_df_private.to_string(index=False))

    results_dir = os.path.join(REPO_ROOT, "results")
    os.makedirs(results_dir, exist_ok=True)
    t0_df_private.to_csv(os.path.join(results_dir, "sa_calibration_t0_cooling_grid_PRIVATE.csv"), index=False)
    print(f"\nPrivate computational T0/cooling grid saved. Full lambda_s sweep follows the same "
          f"pattern (call corrective_search_one_vehicle with trial_config['lambda_s'] varied) -- "
          f"omitted here for brevity; see the PUBLIC-mode cell above for the published reference values.")


## Expected outputs / integrity checks

In [ ]:
checks = {}
if DATA_MODE == "public":
    checks["t0_grid_loaded_nonzero"] = len(t0_df) > 0
    checks["lambda_sweep_loaded_nonzero"] = len(lambda_df) > 0
    checks["selection_reproducible"] = selected_lambda_s == sa_config['lambda_s']
else:
    checks["private_calibration_ran_nonzero_blocks"] = len(calib_blocks) > 0

for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
NOTEBOOK_05_STATUS = "PASS" if all(checks.values()) else "FAIL"
print(f"\nNOTEBOOK 05 STATUS: {NOTEBOOK_05_STATUS}")
assert NOTEBOOK_05_STATUS == "PASS"
